In [ ]:

# 导入必要的库
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# 加载数据
data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/train.csv'
data = pd.read_csv(data_path)

# 查看数据的基本信息
print(data.head())
print(data.info())
print(data.describe())


  surgery  hospital_number  ...  capillary_refill_time     outcome
0     yes           527706  ...             less_3_sec        died
1     yes           528641  ...             less_3_sec       lived
2     yes           535043  ...             more_3_sec  euthanized
3     yes           535043  ...             less_3_sec  euthanized
4     yes           528890  ...             more_3_sec        died

[5 rows x 9 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 986 entries, 0 to 985
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery                986 non-null    object 
 1   hospital_number        986 non-null    int64  
 2   rectal_temp            986 non-null    float64
 3   pulse                  986 non-null    float64
 4   respiratory_rate       986 non-null    float64
 5   peripheral_pulse       938 non-null    object 
 6   mucous_membrane        971 non-null    object 
 7  

In [ ]:

# 处理缺失值
# 对于数值型变量，用中位数填充
data['pulse'].fillna(data['pulse'].median(), inplace=True)
data['rectal_temp'].fillna(data['rectal_temp'].median(), inplace=True)

# 对于分类变量，用众数填充
data['peripheral_pulse'].fillna(data['peripheral_pulse'].mode()[0], inplace=True)
data['mucous_membrane'].fillna(data['mucous_membrane'].mode()[0], inplace=True)

# 分离特征和目标变量
X = data.drop('outcome', axis=1)
y = data['outcome']

# 分割数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 查看分割后的数据
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (788, 8)
X_test shape: (198, 8)
y_train shape: (788,)
y_test shape: (198,)
C:\Users\xuyutian\AppData\Local\Temp\ipykernel_20800\1489646561.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['pulse'].fillna(data['pulse'].median(), inplace=True)
C:\Users\xuyutian\AppData\Local\Temp\ipykernel_20800\1489646561.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will ne

In [ ]:


# 处理缺失值
# 对于数值型变量，用中位数填充
data['pulse'] = data['pulse'].fillna(data['pulse'].median())

data['rectal_temp'] = data['rectal_temp'].fillna(data['rectal_temp'].median())

# 对于分类变量，用众数填充
data['peripheral_pulse'] = data['peripheral_pulse'].fillna(data['peripheral_pulse'].mode()[0])

data['mucous_membrane'] = data['mucous_membrane'].fillna(data['mucous_membrane'].mode()[0])

# 分离特征和目标变量
X = data.drop('outcome', axis=1)
y = data['outcome']

# 分割数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 查看分割后的数据
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")



X_train shape: (788, 8)
X_test shape: (198, 8)
y_train shape: (788,)
y_test shape: (198,)


In [ ]:


# 定义分类和数值型特征
categorical_features = ['surgery', 'peripheral_pulse', 'mucous_membrane', 'capillary_refill_time']
numerical_features = ['rectal_temp', 'pulse', 'respiratory_rate', 'hospital_number']

# 创建预处理管道
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 创建完整流水线，包括预处理和模型训练
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 训练模型
model.fit(X_train, y_train)

# 在测试集上预测
y_pred = model.predict(X_test)

# 计算F1分数
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"F1 Score: {f1:.4f}")



F1 Score: 0.6155


In [ ]:



from sklearn.model_selection import GridSearchCV

# 定义参数网格
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

# 创建GridSearchCV对象
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)

# 进行网格搜索
grid_search.fit(X_train, y_train)

# 输出最佳参数和最佳F1分数
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best F1 Score: {grid_search.best_score_:.4f}")

# 使用最佳参数在测试集上评估模型
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
f1_best = f1_score(y_test, y_pred_best, average='weighted')
print(f"F1 Score on Test Set with Best Parameters: {f1_best:.4f}")




Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best F1 Score: 0.6285
F1 Score on Test Set with Best Parameters: 0.6263
